# Transformer Blocks

Here we will look at what all happends in a tradiational transformer block which forms the foundation of most transformer architectures.

![image.png](transformerBlock.png)
![image.png](transformer.png)

Our process will be:
1. Create embeddings with positional encoding
2. Create a function to compute attention scores.
3. Create an attention layer
4. Create a multihead attention layer
5. Create a transformer block

## Embeddings
Before we get started, we need proper embeddings.

![image.png](encoding.png)

To do this we'll:
1. Tokenize the words
2. Obtain embeddings for tokenized words
3. Get positional encodings
4. Combine these.
   

### Tokenize Data
To start off, we'll use the AutoTokenizer from huggingface.
To use this, you'll likely need to *pip install transformers*
We can then get a pretrained tokenizer and 

In [73]:
from transformers import AutoTokenizer
model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

Next we'll tokenize some text and see what's output...

In [74]:
text = "time flies like an arrow"
# Use tokenizer without automatic tensor conversion (avoids HF torch-version check)
token_ids = tokenizer.encode(text, add_special_tokens=False)
input_ids = torch.tensor([token_ids])  # shape: (batch_size=1, seq_len)
input_ids

tensor([[ 2051, 10029,  2066,  2019,  8612]])

### Token embeddings
Next we'll use a pretrained embedding layer (think the FC you trained in the Word2Vec HW)

In [75]:
from torch import nn
from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_ckpt)
token_emb = nn.Embedding(config.vocab_size, config.hidden_size)

And use it to get a dense vector representation of our tokens.

In [76]:
inputs_embeds = token_emb(input_ids)
inputs_embeds.size()
inputs_embeds

tensor([[[-0.2484,  0.6129, -0.9678,  ..., -0.2848,  0.4628,  1.4510],
         [ 0.1966,  0.9140,  1.2125,  ...,  0.2019, -1.8563,  1.2441],
         [-1.2460, -0.6209, -0.5117,  ..., -0.4079, -0.3702, -1.1794],
         [ 0.3002, -0.6764, -0.6237,  ..., -1.0745, -0.9674,  0.1795],
         [-0.3112,  1.6904, -0.0964,  ...,  0.3680,  1.2147, -0.0479]]],
       grad_fn=<EmbeddingBackward0>)

### Embedding Class
Now let's define an embedding class that gets the embedding for the words and positions, and combines them.
As you'll see there's also a layer norm and dropout layer in this.

In [77]:
import torch
class Embeddings(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embeddings = nn.Embedding(config.vocab_size, 
                                             config.hidden_size)
        self.position_embeddings = nn.Embedding(config.max_position_embeddings,
                                                config.hidden_size)
        self.layer_norm = nn.LayerNorm(config.hidden_size, eps=1e-12)
        self.dropout = nn.Dropout()

    def forward(self, input_ids):
        # Create position IDs for input sequence on the same device as input_ids
        seq_length = input_ids.size(1)
        position_ids = torch.arange(seq_length, dtype=torch.long, device=input_ids.device).unsqueeze(0)
        # Create token and position embeddings
        token_embeddings = self.token_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)
        # Combine token and position embeddings
        embeddings = token_embeddings + position_embeddings
        embeddings = self.layer_norm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

Now we can use this class to get the final embedding:

In [78]:
emb = Embeddings(config)
input_embeds = emb(input_ids)

## Transformer Block
Now it's time to build our transformer block.
To do this we'll first need to define two new layers:
1. Multi-head attention
2. Feed Forward

![image.png](transformerBlock.png)


### Feed Forward Layer
The FeedForward layer is pretty straightforward: is just two fully-connected layers with a ReLu (Gaussian smoothed version of ReLu) between them and a dropout layer on the backend.

In [79]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.linear_1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.linear_2 = nn.Linear(config.intermediate_size, config.hidden_size)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        
    def forward(self, x):
        x = self.linear_1(x)
        x = self.gelu(x)
        x = self.linear_2(x)
        x = self.dropout(x)
        return x

### Multi-Head Attention
Before we can build our multi-head attention layer, we need create attention layers which needs to compute attention.

Recall that we compute attention as:
Let's build this up by first defining attention:
$$attention = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

We'll create a  function for this:

In [80]:
import math
import torch.nn.functional as F
def scaled_dot_product_attention(query, key, value):
    dim_k = query.size(-1)
    scores = torch.bmm(query, key.transpose(1, 2)) / math.sqrt(dim_k)
    weights = F.softmax(scores, dim=-1)
    return torch.bmm(weights, value)

query = key = value = inputs_embeds
scores = scaled_dot_product_attention(query,key,value)
print(scores)

tensor([[[-0.2484,  0.6129, -0.9678,  ..., -0.2848,  0.4628,  1.4510],
         [ 0.1966,  0.9140,  1.2125,  ...,  0.2019, -1.8563,  1.2441],
         [-1.2460, -0.6209, -0.5117,  ..., -0.4079, -0.3702, -1.1794],
         [ 0.3002, -0.6764, -0.6237,  ..., -1.0745, -0.9674,  0.1795],
         [-0.3112,  1.6904, -0.0964,  ...,  0.3680,  1.2147, -0.0479]]],
       grad_fn=<BmmBackward0>)


#### Attention Head
To create our Multi-Head Attention layer, let's first create an *Attention Head*.

An attention head just computes attention, but where the query, key, and value all have a fully connected layer associated with them:

In [81]:
class AttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim):
        super().__init__()
        self.q = nn.Linear(embed_dim, head_dim)
        self.k = nn.Linear(embed_dim, head_dim)
        self.v = nn.Linear(embed_dim, head_dim)

    def forward(self, hidden_state):
        attn_outputs = scaled_dot_product_attention(
            self.q(hidden_state), self.k(hidden_state), self.v(hidden_state))
        return attn_outputs

Now we can create a mult-head attention layer!

In [82]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        embed_dim = config.hidden_size
        num_heads = config.num_attention_heads
        head_dim = embed_dim // num_heads
        self.heads = nn.ModuleList(
            [AttentionHead(embed_dim, head_dim) for _ in range(num_heads)]
        )
        self.output_linear = nn.Linear(embed_dim, embed_dim)

    def forward(self, hidden_state):
        x = torch.cat([h(hidden_state) for h in self.heads], dim=-1)
        x = self.output_linear(x)
        return x

### Transformer Encoder Block
Now we can create a transformer encoder block!

As the figure below demonstrates this includes:
1. Mult-head attention
2. Skip (residual) connections
3. Feed forward layer

![image.png](transformerBlock.png)

In [83]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(config.hidden_size)
        self.layer_norm_2 = nn.LayerNorm(config.hidden_size)
        self.attention = MultiHeadAttention(config)
        self.feed_forward = FeedForward(config)

    def forward(self, x):
        # Apply layer normalization and then copy input into query, key, value
        hidden_state = self.layer_norm_1(x)
        # Apply attention with a skip connection
        x = x + self.attention(hidden_state)
        # Apply feed-forward layer with a skip connection
        x = x + self.feed_forward(self.layer_norm_2(x))
        return x

### Forward Propagate through A Transformer Encoder Layer!

In [84]:
encoder_layer = TransformerEncoderLayer(config)
encoding = encoder_layer(input_embeds)
print(encoding)


tensor([[[ 0.0671,  0.8578, -1.1110,  ...,  0.4278, -0.1612, -0.3187],
         [ 0.0559, -2.9445,  1.0953,  ...,  0.2103,  0.0099, -1.1507],
         [-1.2226,  0.2368, -0.1662,  ...,  0.2445, -0.0566, -0.1774],
         [ 2.8400,  2.6009,  3.0877,  ..., -4.0658, -0.4692, -0.1423],
         [-0.2537, -0.0354,  0.4109,  ..., -0.1658, -0.2586,  0.1855]]],
       grad_fn=<AddBackward0>)


## Transform Encoder
If we like, we can encapsilate the entire process in another class, and support multiple transformer encoding blocks.

In [85]:
class TransformerEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embeddings = Embeddings(config)
        self.layers = nn.ModuleList([TransformerEncoderLayer(config) 
                                     for _ in range(config.num_hidden_layers)])

    def forward(self, x):
        x = self.embeddings(x)
        for layer in self.layers:
            x = layer(x)
        return x
        
encoder = TransformerEncoder(config)
encoding = encoder(input_ids)
print(encoding)

tensor([[[-0.7172, -0.3093, -0.4473,  ...,  0.1096, -1.8058,  0.5005],
         [-0.4965,  0.7870, -0.2488,  ...,  0.2719, -2.7124, -1.8880],
         [-0.5982,  0.2333,  1.8938,  ..., -1.0401,  2.3257, -0.8482],
         [ 0.3893,  2.9618, -0.2499,  ..., -2.1466,  1.6396,  0.4404],
         [-0.3185,  2.2156, -1.3599,  ...,  2.3341, -1.2729, -1.8999]]],
       grad_fn=<AddBackward0>)


## Classification
If we then wanted to use this for classification, we could add a dropout layer then a fully connected layer to take us to final target number of labels

In [86]:
class TransformerForSequenceClassification(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.encoder = TransformerEncoder(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        
    def forward(self, x):
        x = self.encoder(x)[:, 0, :] # select hidden state of [CLS] token
        x = self.dropout(x)
        x = self.classifier(x)
        return x

In [87]:
config.num_labels = 3
encoder_classifier = TransformerForSequenceClassification(config)
print(encoder_classifier(input_ids))

tensor([[ 0.5338,  2.0832, -0.2649]], grad_fn=<AddmmBackward0>)


## Toy Training Example

Now that we have a `TransformerForSequenceClassification`, we can run a tiny training loop on a small toy dataset just to see the loss go down. This is *not* meant to be a good model or dataset, just a demonstration of how training would plug into the architecture we built above.

In [90]:
# Simple toy dataset: 3 texts, 3 labels
train_texts = [
    "time flies like an arrow",    # class 0
    "fruit flies like a banana",   # class 1
    "the cat sat on the mat",      # class 2
]
train_labels = torch.tensor([0, 1, 2])

# Tokenize without using return_tensors to avoid HF torch checks
train_encodings = tokenizer(train_texts, add_special_tokens=False, padding=True, truncation=True)
train_input_ids = torch.tensor(train_encodings["input_ids"])

# Move model and data to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Model is trained on {device}')
encoder_classifier.to(device)
train_input_ids = train_input_ids.to(device)
train_labels = train_labels.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(encoder_classifier.parameters(), lr=1e-4)

# Tiny training loop (just for demonstration)
num_epochs = 20
for epoch in range(num_epochs):
    encoder_classifier.train()
    optimizer.zero_grad()

    logits = encoder_classifier(train_input_ids)
    loss = criterion(logits, train_labels)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        preds = logits.argmax(dim=-1)
        acc = (preds == train_labels).float().mean().item()
        print(f"Epoch {epoch+1}/{num_epochs} - loss: {loss.item():.4f}, acc: {acc:.2f}")

print("Final logits:", logits.detach().cpu())
print("Predicted labels:", preds.detach().cpu())

Model is trained on cuda
Epoch 5/20 - loss: 0.0000, acc: 1.00
Epoch 10/20 - loss: 0.0000, acc: 1.00
Epoch 15/20 - loss: 0.0000, acc: 1.00
Epoch 20/20 - loss: 0.0000, acc: 1.00
Final logits: tensor([[ 19.4808, -11.2560,  -5.0406],
        [-12.1657,  14.6042,   0.0869],
        [-14.1864,  -6.6363,  20.4441]])
Predicted labels: tensor([0, 1, 2])


## Heavier Training Benchmark (GPU Stress Test)

Below is a synthetic training loop that runs the transformer classifier on **random token sequences** with many samples, longer sequences, and multiple epochs. This is not for accuracy (the labels are random), but to **push your GPU** and report rough tokens/second throughput.

You can increase `num_samples`, `seq_length`, `batch_size`, or `num_epochs` to make it more intensive (watch GPU memory usage).

The first cell measures **training throughput** (forward + backward). A second cell below will measure **pure inference throughput** (forward-only).

In [91]:
import time

# Stress-test configuration (tweak these to increase/decrease load)
num_samples = 4096   # total synthetic examples
seq_length = 64      # tokens per example
batch_size = 32
num_epochs = 5

# Use same device as before
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder_classifier.to(device)
encoder_classifier.train()

vocab_size = config.vocab_size
num_labels = config.num_labels

# Synthetic random data entirely on the GPU/CPU device
torch.manual_seed(0)
synthetic_input_ids = torch.randint(
    low=0,
    high=vocab_size,
    size=(num_samples, seq_length),
    dtype=torch.long,
    device=device,
)
synthetic_labels = torch.randint(
    low=0,
    high=num_labels,
    size=(num_samples,),
    dtype=torch.long,
    device=device,
)

# Simple DataLoader over the in-memory tensors
synthetic_dataset = list(zip(synthetic_input_ids, synthetic_labels))
loader = torch.utils.data.DataLoader(
    synthetic_dataset,
    batch_size=batch_size,
    shuffle=True,
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(encoder_classifier.parameters(), lr=3e-5)

# Optional warm-up
if device.type == "cuda":
    torch.cuda.synchronize()

start = time.time()
for epoch in range(num_epochs):
    epoch_loss = 0.0
    total = 0
    correct = 0
    for batch_input_ids, batch_labels in loader:
        # batch_* are already on the correct device
        optimizer.zero_grad()
        logits = encoder_classifier(batch_input_ids)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * batch_input_ids.size(0)
        preds = logits.argmax(dim=-1)
        total += batch_labels.size(0)
        correct += (preds == batch_labels).sum().item()

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed_epoch = time.time() - start
    tokens_so_far = (epoch + 1) * num_samples * seq_length
    tokens_per_sec_epoch = tokens_so_far / elapsed_epoch
    print(
        f"Epoch {epoch+1}/{num_epochs} - loss: {epoch_loss/total:.4f}, "
        f"acc: {correct/total:.3f}, time_since_start: {elapsed_epoch:.2f}s, "
        f"avg_tokens/sec: {tokens_per_sec_epoch:.1f}"
    )

elapsed = time.time() - start
processed_tokens = num_samples * seq_length * num_epochs
print(
    f"Processed {processed_tokens} tokens in {elapsed:.2f}s "
    f"(~{processed_tokens/elapsed:.1f} tokens/sec)"
)

/opt/rocm-pytorch/lib/python3.11/site-packages/torch/cuda/__init__.py:546: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Epoch 1/5 - loss: 1.5475, acc: 0.327, time_since_start: 38.24s
Epoch 2/5 - loss: 1.1966, acc: 0.368, time_since_start: 76.56s
Epoch 3/5 - loss: 1.1538, acc: 0.390, time_since_start: 114.91s
Epoch 4/5 - loss: 1.1341, acc: 0.403, time_since_start: 153.28s
Epoch 5/5 - loss: 1.1146, acc: 0.408, time_since_start: 191.66s
Processed 1310720 tokens in 191.66s (~6838.6 tokens/sec)


## Pure Inference Throughput Benchmark

This benchmark measures **forward-only** throughput (no backward pass) on random inputs to approximate maximum inference speed. Adjust `num_batches`, `batch_size`, and `seq_length` as desired.

In [92]:
import time

# Inference benchmark configuration
inf_num_batches = 200
inf_batch_size = 32
inf_seq_length = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder_classifier.to(device)
encoder_classifier.eval()

vocab_size = config.vocab_size

# Pre-create one random batch to avoid data overhead in loop
torch.manual_seed(123)
inf_input_ids = torch.randint(
    low=0,
    high=vocab_size,
    size=(inf_batch_size, inf_seq_length),
    dtype=torch.long,
    device=device,
)

# Warm-up
with torch.no_grad():
    _ = encoder_classifier(inf_input_ids)
if device.type == "cuda":
    torch.cuda.synchronize()

start = time.time()
with torch.no_grad():
    for _ in range(inf_num_batches):
        _ = encoder_classifier(inf_input_ids)
if device.type == "cuda":
    torch.cuda.synchronize()

elapsed = time.time() - start
processed_tokens = inf_num_batches * inf_batch_size * inf_seq_length
print(
    f"Inference: processed {processed_tokens} tokens in {elapsed:.2f}s "
    f"(~{processed_tokens/elapsed:.1f} tokens/sec)"
)

Inference: processed 409600 tokens in 20.03s (~20447.8 tokens/sec)
